# 🧠 Transformer (下)：大模型的“个性控制中心”

湖北理工学院《人工智能理论》课程资料

📝 **本节核心命题：如何精准调控模型的输出风格？**

在前面的章节中，我们理解了 Transformer 如何生成概率分布。但在实际应用中，我们并不总是只取概率最大的那个词。通过 **Temperature**、**Top-K** 和 **Top-P**，我们可以像调音师一样，旋转旋钮来改变大模型的性格：是“严谨好学”还是“天马行空”？

**⚠️ 实验原则：本节将采用“变量控制法”，每次只深入讨论和实验一个参数，保持其他参数为中性值，以观察最纯粹的影响。**

---

## 📊 1. 复习：从 Logits 到概率分布

模型最后一层输出的是非标准化的分值（Logits）。Softmax 会将它们映射为概率：

$$P(y_i) = \frac{e^{z_i}}{\sum_j e^{z_j}}$$

我们先准备好全局通用的实验样本：一组模拟的城市续写分值。

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from openai import OpenAI

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

# 核心：设置 Pandas 不截断长文本，确保能看到完整的续写结果
pd.set_option('display.max_colwidth', None)

# 1. 模拟 10 个候选 Token 的 Logits
vocab = ["北京", "上海", "广州", "深圳", "杭州", "成都", "武汉", "西安", "南京", "天津"]
mock_logits = torch.tensor([5.5, 4.8, 4.0, 3.8, 3.5, 3.2, 3.0, 2.5, 2.0, 1.0])

# 2. API 配置 (继承您的设置)
API_KEY = ""
BASE_URL = "https://dashscope.aliyuncs.com/compatible-mode/v1"
MODEL_NAME = "qwen3.5-flash"

client = OpenAI(api_key=API_KEY, base_url=BASE_URL)

def get_api_response(prompt, t=1.0, k=None, p=1.0):
    """
    统一 API 调用函数，封装了参数逻辑
    """
    try:
        params = {
            "model": MODEL_NAME,
            "messages": [{"role": "user", "content": prompt}],
            "temperature": t,
            "top_p": p,
            "max_tokens": 150 # 增加输出长度以便观察完整风格
        }
        # 只有在设置了 top_k 且 Provider 支持时添加
        if k is not None:
            params["extra_body"] = {"top_k": k}
            
        response = client.chat.completions.create(**params)
        return response.choices[0].message.content.strip()
    except Exception as e:
        return f"❌ 调用失败: {e}"

print("✅ 实验环境已初始化完毕。")

## 🌡️ 2. Temperature (温度)

### 数学原理：Logits 的缩放

Temperature $T$ 在 Softmax 之前介入：

$$P(y_i) = \frac{e^{z_i/T}}{\sum_j e^{z_j/T}}$$

*   **$T \to 0$ (低温)**：放大分值差异。极小概率的词被几乎削减为 0，高概率的词占据统治地位（倾向于确定性输出）。
*   **$T = 1.0$ (中性)**：不改变 Logits 比例。
*   **$T > 1.0$ (高温)**：缩小分值差异。分布变得平缓，低概率词获得更多“露脸”机会（增加随机性和多样性）。

In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"]="TRUE"

def plot_t_distribution(logits):
    temps = [0.1, 0.7, 1.5]
    plt.figure(figsize=(15, 4))
    
    for i, t in enumerate(temps):
        probs = F.softmax(logits / t, dim=0)
        plt.subplot(1, 3, i + 1)
        plt.bar(vocab, probs.numpy(), color='royalblue', alpha=0.8)
        plt.title(f"Temperature = {t}")
        plt.ylim(0, 1.0)
        plt.ylabel("概率")
        plt.xticks(rotation=45)
    
    plt.tight_layout()
    plt.show()

plot_t_distribution(mock_logits)
print("🎓 观察：当 T=0.1 时，‘北京’几乎夺走了 100% 的概率；当 T=1.5 时，各城市几乎平起平坐。")

### Standalone Lab: 纯温度实验

我们固定 $Top-K=Vocab$（不截断）和 $Top-P=1.0$（不截断），仅改变 `temperature`。

In [ ]:
prompt_t = "用诗意的语言续写（直接续写不给选择）：秋天的雨落在了..."

t_configs = [
    {"name": "极端稳健", "T": 0.01},
    {"name": "默认平衡", "T": 0.7},
    {"name": "极其跳跃", "T": 1.5}
]

t_results = []
for cfg in t_configs:
    print(f"🚀 正在测试 Temperature={cfg['T']}...")
    ans = get_api_response(prompt_t, t=cfg['T'], p=1.0) # 保持 P=1.0 为中性
    t_results.append([cfg['name'], cfg['T'], ans])

df_t = pd.DataFrame(t_results, columns=["风格", "T值", "模型续写结果"])
# 💡 使用 Style 让 \n 在表格中正常换行，并允许显示完整内容
display(df_t.style.set_properties(**{'white-space': 'pre-wrap', 'text-align': 'left'}))

## ✂️ 3. Top-K Sampling

### 理论：固定数量的“大剪刀”

**Top-K** 逻辑非常简单：
1. 将所有 Token 按概率从大到小排序。
2. 仅保留前 $K$ 个。 
3. 将第 $K+1$ 到最后的所有概率归零，并对前 $K$ 个重新归一化。

**它的作用：** 彻底杜绝那些“长尾”里面的离谱词汇，即使在高温下也能保证不胡言乱语。

In [ ]:
def plot_top_k(logits, k_val):
    probs = F.softmax(logits, dim=0)
    
    # 找到前 K 个的索引
    top_vals, top_idx = torch.topk(probs, k_val)
    
    colors = ['forestgreen' if i in top_idx else 'lightgrey' for i in range(len(probs))]
    
    plt.figure(figsize=(10, 4))
    plt.bar(vocab, probs.numpy(), color=colors)
    plt.title(f"Top-K Sampling (K={k_val}) 示意图")
    plt.axvline(x=k_val-0.5, color='red', linestyle='--', label=f"K={k_val} 剪断线")
    plt.legend()
    plt.show()

plot_top_k(mock_logits, k_val=3)
print("🎓 观察：只有前 3 名有资格参与下一轮‘掷骰子’，灰色的词被剥夺了发言权。")

### Standalone Lab: 纯 Top-K 实验

我们固定 $Temperature=1.0$（不缩放）和 $Top-P=1.0$（不截断），改变 `top_k`。

In [ ]:
prompt_k = "推荐 3 本关于人工智能的书："

k_configs = [
    {"name": "极致单选", "K": 1},
    {"name": "稳健精选", "K": 5},
    {"name": "大浪淘沙", "K": 50}
]

k_results = []
for cfg in k_configs:
    print(f"🚀 正在测试 Top-K={cfg['K']}...")
    ans = get_api_response(prompt_k, t=1.0, k=cfg['K'], p=1.0)
    k_results.append([cfg['name'], cfg['K'], ans])

df_k = pd.DataFrame(k_results, columns=["策略", "K值", "模型续写结果"])
display(df_k.style.set_properties(**{'white-space': 'pre-wrap', 'text-align': 'left'}))

## 🎯 4. Top-P (Nucleus Sampling)

### 理论：动态的“能量球”

**Top-P** 与 Top-K 的区别在于它是**动态**的。它不再规定选几个次，而是规定选出的词的**累积概率之和**必须达到 $P$。

*   如果模型非常有信心（概率集中在 1-2 个词上），它可能只在 2 个词里选。
*   如果模型很困惑（概率非常分散），它会自动扩大选择范围，包含更多词，直到总概率达到 $P$。

**优势：** 这种采样方式被称为“核心采样（Nucleus Sampling）”，它比 Top-K 更聪明，能根据上下文自动调整“脑洞”大小。

In [ ]:
def plot_top_p(logits, p_val):
    probs = F.softmax(logits, dim=0)
    sorted_probs, sorted_indices = torch.sort(probs, descending=True)
    cumulative_probs = torch.cumsum(sorted_probs, dim=0)
    
    # 找到第一个累积概率 >= p 的索引
    cutoff = (cumulative_probs >= p_val).nonzero()[0].item()
    
    plt.figure(figsize=(10, 4))
    plt.step(range(len(cumulative_probs)), cumulative_probs.numpy(), where='post', label='累积概率曲线')
    plt.axhline(y=p_val, color='r', linestyle='--', label=f'Top-P 切线 (P={p_val})')
    plt.fill_between(range(cutoff + 1), cumulative_probs[:cutoff+1].numpy(), alpha=0.2, color='orange', label='采样核心区间')
    plt.xlabel("按概率排序的 Token 索引")
    plt.ylabel("累积概率")
    plt.legend()
    plt.title("Top-P (Nucleus Sampling) 动态边界图")
    plt.show()

plot_top_p(mock_logits, p_val=0.9)
print("🎓 观察：当累积概率攒够 0.9 时，后面的 Token 就不带它们玩了。")

### Standalone Lab: 纯 Top-P 实验

我们固定 $Temperature=1.0$（不缩放）和 $Top-K=None$（不采用静态过滤），改变 `top_p`。

In [ ]:
prompt_p = "如果你是一个旅行博主，写一句话描述大理的风景："

p_configs = [
    {"name": "极度专注", "P": 0.1},
    {"name": "核心精华", "P": 0.85},
    {"name": "全量包容", "P": 1.0}
]

p_results = []
for cfg in p_configs:
    print(f"🚀 正在测试 Top-P={cfg['P']}...")
    ans = get_api_response(prompt_p, t=1.0, p=cfg['P'])
    p_results.append([cfg['name'], cfg['P'], ans])

df_p = pd.DataFrame(p_results, columns=["策略", "P值", "模型续写结果"])
display(df_p.style.set_properties(**{'white-space': 'pre-wrap', 'text-align': 'left'}))

## 🎓 5. 总结：三大参数的“博弈位点”

| 参数 | 调节对象 | 核心逻辑 | 典型应用 |
| :--- | :--- | :--- | :--- |
| **Temperature** | 概率分布的**形态** | 放大/缩小分值差距 | 决定总体的“创造力”活跃度 |
| **Top-K** | 候选词的**数量** | 粗暴截除低分后的尾巴 | 适合固定字典、小词表的稳定性任务 |
| **Top-P** | 候选词的**能量总和** | 根据概率动态划定范围 | 现代对话模型、长文本生成的标配 |

💡 **实战秘籍**：
*   通常我们将 $T$ 设在 **0.7 - 0.8** 之间以寻找默认平衡。
*   在大多数现代库（如 HuggingFace 或 OpenAI API）中，虽然可以同时设置，但建议**优先调整一个**。如果调了 $T$，则令 $P=1, K=None$；反之亦然。
*   听说Top-K可能不用了，有兴趣的同学自行求证